In [ ]:
import numpy as np
import xarray as xr
import pandas as pd
from scipy import signal
from scipy.ndimage import center_of_mass
from scipy.signal import hilbert, correlate
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter

In [ ]:
def bandpass_for_propagation(ssta, dt_months=1, 
                              low_period=3,   # 最短周期（月）
                              high_period=7*12): # 最长周期（月）
    nt, nx = ssta.shape
    
    # 采样频率（周期/月）
    fs = 1.0 / dt_months
    
    # 转换为频率
    low_freq = 1.0 / high_period   # 低频截止
    high_freq = 1.0 / low_period   # 高频截止
    
    # 对每个经度点做带通滤波
    ssta_filtered = np.zeros_like(ssta)
    
    for i in range(nx):
        # Butterworth带通滤波
        sos = signal.butter(4, [low_freq, high_freq], 
                           btype='band', fs=fs, output='sos')
        ssta_filtered[:, i] = signal.sosfiltfilt(sos, ssta[:, i])
    
    return ssta_filtered

In [ ]:
xr.open_dataset(r'C:\Users\jiuwe\study\data\SST\HadISST_sst.nc')

In [ ]:
lat = np.arange(-90, 91)
lon = np.arange(0, 361)
ds_sst = xr.open_dataset(r'C:\Users\jiuwe\study\data\SST\HadISST_sst.nc').rename({'latitude':'lat', 'longitude':'lon'})
sst_org = ds_sst.sst.loc['1870':'2025']
s1 = sst_org.loc[:, :, -180:0]
s2 = sst_org.loc[:, :, -0:180]
s1 = s1.assign_coords(lon=s1.lon.values+360)
sst = xr.concat([s2, s1], dim='lon')
# sst.assign_coords(time=pd.date_range('1870-01-01', '2025-12-01', freq='MS')).interp(lat=lat, lon=lon).to_netcdf('./data/sst.nc')
sst_cli = sst.loc['1991':'2020']
sst_cli = sst_cli.groupby(sst_cli.time.dt.month).mean()
sst_ano = sst.groupby(sst.time.dt.month) - sst_cli
lat = np.arange(-90, 91)
lon = np.arange(0, 361)
sst_ano.assign_coords(time=pd.date_range('1870-01-01', '2025-12-01', freq='MS')).interp(lat=lat, lon=lon).to_netcdf('./data/ssta.nc')

In [ ]:
sst = xr.open_dataset('sst.nc').sel(lon=slice(120, 270)).sst
sst_cli = sst.loc['1991':'2020']
sst_cli = sst_cli.groupby(sst_cli.time.dt.month).mean()
sst_ano = sst.groupby(sst.time.dt.month) - sst_cli
sst_ano = sst_ano.loc['1950':'2024']
time = sst_ano.time
lon = sst_ano.lon
# sst_an = bandpass_for_propagation(sst_ano.loc[:, 5:-5, :].mean(dim='lat'))
# sst_an = xr.DataArray(sst_an, coords={'time':time, 'lon':lon})
sst_an = sst_ano.loc[:, 5:-5, :].mean(dim='lat')
nino = sst_ano.loc[:, 5:-5, 120:240].mean(dim=['lat', 'lon']).rolling(time=3, center=True, min_periods=1).mean()
nt = len(time)


In [ ]:
t = 0
start = 0
end = 0
dev = nino.copy()
dev[:] = np.nan
events = []
while t < nt-1:
    if nino[t] <= 0 and nino[t+1] > 0:
        zero = t
    if nino[t] <= 0.5 and nino[t+1] > 0.5:
        start = t
    if nino[t] >= 0.5 and nino[t+1] < 0.5:
        end = t
    if end > start:
        max_loc = np.argmax(nino[zero:end+1].values)
        dev[zero-1:zero+max_loc+2] = nino[zero-1:zero+max_loc+2]
        events.append((zero-1, zero+max_loc+2))
        print(time[zero].values, time[zero+max_loc].values, end-start+1)
        end = start
    t += 1
events = np.array(events)

In [ ]:
start_date = pd.to_datetime(time[events[:, 0]].values)
end_date = pd.to_datetime(time[events[:, 1]].values)
El_Nino = []
for i in range(events.shape[0]):
    year0 = start_date[i].year
    year1 = end_date[i].year
    month0 = start_date[i].month
    month1 = end_date[i].month
    
    if month1 > 9:
        month1_ = 3
        year1_ = year1 + 1
    elif month1 < 4:
        month1_ = 3
        year1_ = year1
    else:
        continue
    month0_ = 4
    year0_ = year1_ - 1
    # print(year0, month0, year1, month1)
    # print(year0_, month0_, year1_, month1_)
    # print('-------')
    El_Nino.append([f'{year0_}-{month0_:02d}', f'{year1_}-{month1_:02d}'])
        
El_Nino = np.array(El_Nino, dtype='datetime64[M]')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2))
ax.plot(time, nino)
ax.plot(time, dev)


In [ ]:
tt = 12
start = events[tt][0]
end = events[tt][1]
sst_a = sst_an[start:end]
ssta_pos = sst_a.where(sst_a>=0.6)
# ssta = sst_a - sst_a.mean(dim='lon')
# ssta = ssta.where(sst_a>=0, 0)
print(time[start].values, time[end].values)

In [ ]:
start = '1996-10'
end = '1997-12'
sst_a = sst_an.loc[start:end]


In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
sst_a.plot.contourf(levels=np.linspace(-3, 3, 21), cmap='bwr', ax=ax)
sst_a.plot.contour(levels=[0.5, 1, 1.5], cmap='w', ax=ax)

In [ ]:
sst_ano[start:end].plot.contourf(col='time', col_wrap=4, cmap='bwr', size=2, aspect=2, levels=21)

In [ ]:
Rp = nino3+nino4
Ip = nino3-nino4
UCEI = (nino3+nino4) + (nino3-nino4)*1j
r = abs(UCEI)
theta = np.arctan2(nino3-nino4, nino3+nino4)
theta = np.where(theta<0, theta+2*np.pi, theta)
the1 = 15*np.pi/180
the2 = np.pi-the1
the3 = np.pi+the1
the4 = 2*np.pi-the1

typ = np.zeros((len(r)), dtype='int')
typ[np.logical_and(theta>the1, theta<np.pi/2)] = 0
typ[np.logical_and(theta>0, theta<the1)] = 1
typ[np.logical_and(theta>the4, theta<2*np.pi)] = 1
typ[np.logical_and(theta>1.5*np.pi, theta<the4)] = 2
typ[np.logical_and(theta>the3, theta<1.5*np.pi)] = 3
typ[np.logical_and(theta>the2, theta<the3)] = 4
typ[np.logical_and(theta>0.5*np.pi, theta<the2)] = 5

ints_lev = [0, 0.5, 1, 2, 5]

In [ ]:
UCEI

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
xx = 5*np.cos(15*np.pi/180)
x = np.linspace(-xx, xx, 50)
y1 = x*np.tan(15*np.pi/180)
y2 = x*np.tan(-15*np.pi/180)
ax.plot(x, y1, c='k')
ax.plot(x, y2, c='k')

colors = np.empty((len(r)), dtype='str')
colors[typ==0] = 'r'
colors[typ==1] = 'y'
colors[typ==2] = 'm'
colors[typ==3] = 'b'
colors[typ==4] = 'g'
colors[typ==5] = 'c'
ax.scatter(nino3+nino4, nino3-nino4, s=2, c=colors.tolist())
for i in range(len(r)-1):
    ax.plot(Rp[i:i+2], Ip[i:i+2], c=colors[i])
    
for i in range(5):
    circle = Circle((0, 0), radius=ints_lev[i], fill=False, edgecolor='k', linestyle='-', linewidth=0.9, alpha=0.8)
    ax.add_patch(circle)

ax.spines['left'].set_position('zero')  
ax.spines['bottom'].set_position('zero') 

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.xaxis.set_ticks_position('bottom')
ax.yaxis.set_ticks_position('left')

ax.set_ylim([-5, 5])
ax.set_xlim([-5, 5])

In [ ]:
colors

In [ ]:
sst_ano.loc[start:end].plot.contourf(col='time', col_wrap=4, cmap='bwr', size=2, aspect=2, levels=21)

In [ ]:
sst_ano.loc['2004-05':'2005-08'].mean(dim='time').plot.contourf(cmap='bwr', size=2, aspect=2, levels=21)

In [ ]:
sst_ano.loc['1997-05':'1998-08'].mean(dim='time').plot.contourf(cmap='bwr', size=2, aspect=2, levels=21)

In [ ]:
xr.open_dataset('../../data/necp/monthly/uwnd.mon.mean.nc')

In [ ]:
938/12